This cell initializes the analysis by loading pre-computed data artifacts necessary for examining the temporal dynamics of AI-related concerns and benefits in public dialogues. By retrieving processed outputs, including embeddings, cluster labels, and summary data, it ensures that subsequent analyses can be conducted efficiently without the need to re-run computationally intensive tasks, thereby streamlining the exploration of trends over time.

In [ ]:
# @title Load pre-computed artifacts
# Run this cell before any analysis cell.  It loads all outputs written by
# 01_processing.ipynb from disk so this notebook never calls the OpenAI API
# or re-runs k-means.
from pathlib import Path
import pub_dialogue.utils as du
from pub_dialogue.utils import show_status, show_complete
import pandas as pd
import numpy as np

OUTPUT_FOLDER     = Path("outputs")
CHECKPOINT_FOLDER = Path("checkpoints")

a = du.load_artifacts(OUTPUT_FOLDER, CHECKPOINT_FOLDER)

chunks_df    = a["chunks_df"]
concerns_df  = a["concerns_df"]
benefits_df  = a["benefits_df"]

concern_embeddings  = a["concern_embeddings"]
benefit_embeddings  = a["benefit_embeddings"]
concern_centroids   = a["concern_centroids"]
benefit_centroids   = a["benefit_centroids"]

concern_ids          = a["concern_ids"]
benefit_ids          = a["benefit_ids"]
cluster_labels       = a["cluster_labels"]
benefit_cluster_labels = a["benefit_cluster_labels"]
cluster_summary_df   = a["cluster_summary_df"]
benefit_cluster_summary_df = a["benefit_cluster_summary_df"]

framing_lens_mappings         = a["framing_lens_mappings"]
benefit_framing_lens_mappings = a["benefit_framing_lens_mappings"]

cluster_entropy           = a["cluster_entropy"]
cluster_entropy_norm      = a["cluster_entropy_norm"]
cross_cutting_clusters    = a["cross_cutting_clusters"]

benefit_cluster_entropy          = a["benefit_cluster_entropy"]
normalized_entropy_benefits      = a["normalized_entropy_benefits"]
cross_cutting_clusters_benefits  = a["cross_cutting_clusters_benefits"]

# Convenience: CLUSTER_LABELS / BENEFIT_CLUSTER_LABELS dicts used by plots
CLUSTER_LABELS        = {int(k): v for k, v in cluster_labels.items()}
BENEFIT_CLUSTER_LABELS = {int(k): v for k, v in benefit_cluster_labels.items()}

# Re-apply technology_meta merge if not already present in the loaded CSV
if "technology_meta" not in concerns_df.columns:
    import pandas as pd
    _tech = chunks_df[["chunk_id", "technology_meta"]]
    concerns_df = concerns_df.merge(_tech, on="chunk_id", how="left")
    benefits_df = benefits_df.merge(_tech, on="chunk_id", how="left")

print(f"Artifacts loaded from {OUTPUT_FOLDER} / {CHECKPOINT_FOLDER}")
print(f"  Chunks: {len(chunks_df):,}  |  Concerns: {len(concerns_df):,}  |  Benefits: {len(benefits_df):,}")

This cell imports essential functions for calculating metrics such as normalized entropy, the Herfindahl-Hirschman Index (HHI), and top-k share, which are crucial for analyzing the distribution and concentration of AI-related concerns and benefits in public dialogues over time. By leveraging these metrics, the analysis can reveal shifts in public sentiment and focus, providing insights into how societal perceptions of AI evolve across different periods.

In [ ]:
# normalized_entropy, hhi, topk_share, parse_year — imported from dialogue_utils (see Cell 5)
pass

This cell conducts an analysis of AI-related concerns by aggregating data into broader time windows, addressing the sparsity of year-by-year data that can skew entropy calculations. By grouping the data into four wider intervals, it ensures that the sample sizes are sufficient to produce meaningful insights into the diversity of concerns over time, thus enhancing the interpretability of the results and supporting more robust temporal trend claims in the overall study.

In [ ]:
# @title (v16) Wider-time-window AI concern analysis
# Issue 4 partial fix from the audit: year-by-year AI concern data is
# extremely sparse for some years (e.g. 2024 had 9 phrases in v15). With
# small N, normalised entropy is mathematically forced toward its maximum
# regardless of the underlying distribution, confounding any temporal-trend
# claim about "concern diversity" with sample size.
#
# This cell repeats the temporal AI analysis using 4 wider windows
# (2004-2017, 2018-2020, 2021-2023, 2024-2025) so each window contains
# enough data to make the entropy/share numbers interpretable. The
# year-by-year analysis is retained above; treat this as the primary table
# for any quantitative temporal claim in the paper.

show_status("Running wider-window temporal AI analysis...")

from scipy.stats import entropy
from pub_dialogue.address import assign_window

ai_concerns_window = concerns_df[concerns_df["technology_meta"] == "AI"].copy()
ai_concerns_window["time_window"] = ai_concerns_window["year"].apply(assign_window)
ai_concerns_window = ai_concerns_window.dropna(subset=["time_window"])

# Per-window: total phrases, cluster distribution, entropy
window_summary = []
for window, group in ai_concerns_window.groupby("time_window"):
    n_phrases = len(group)
    cluster_counts = group["cluster_id"].value_counts()
    probs = (cluster_counts / cluster_counts.sum()).values if cluster_counts.sum() > 0 else np.array([])
    raw_ent = float(entropy(probs)) if len(probs) > 1 else 0.0
    norm_ent = raw_ent / np.log(len(cluster_counts)) if len(cluster_counts) > 1 else 0.0
    window_summary.append({
        "time_window": window,
        "n_phrases": n_phrases,
        "n_clusters_present": int((cluster_counts > 0).sum()),
        "n_documents": int(group["source_file"].nunique()),
        "raw_entropy": raw_ent,
        "normalized_entropy_within_window": norm_ent,
    })

window_summary_df = pd.DataFrame(window_summary).sort_values("time_window")
window_summary_df.to_csv(OUTPUT_FOLDER / "ai_concern_window_summary.csv", index=False)

print("Wider-window AI concern summary:")
print(window_summary_df.to_string(index=False))
print()
print("Note: 'normalized_entropy_within_window' is normalised by the number of")
print("clusters that contain any phrases in that window — so it is bounded by")
print("the data, not by the absolute cluster count. With more data per window,")
print("these values are more interpretable than the year-by-year ones above.")

# Also: per-window top clusters (qualitative)
top_clusters_per_window = []
for window, group in ai_concerns_window.groupby("time_window"):
    counts = group["cluster_id"].value_counts().head(10)
    for cid, n in counts.items():
        label = CLUSTER_LABELS.get(cid, f"Cluster {cid}") if "CLUSTER_LABELS" in globals() else f"Cluster {cid}"
        top_clusters_per_window.append({
            "time_window": window,
            "cluster_id": cid,
            "label": label,
            "n_phrases": int(n),
            "share_within_window": float(n) / len(group),
        })

top_window_df = pd.DataFrame(top_clusters_per_window)
top_window_df.to_csv(OUTPUT_FOLDER / "ai_concern_window_top_clusters.csv", index=False)
print(f"\nWrote top-10-clusters-per-window table to ai_concern_window_top_clusters.csv")

show_complete("Wider-window analysis complete")

This cell visualizes the trajectory of AI-related concerns over time by plotting the displacement of these concerns in a two-dimensional principal component space. By centering the trajectory on the initial year, it allows for an intuitive understanding of how public sentiment regarding AI has evolved, highlighting shifts in concern levels across different time periods. This analysis is crucial for identifying trends and patterns in public dialogue, which can inform future discussions and policy-making related to AI technologies.

In [ ]:
# @title Compute AI concern PCA trajectory (traj)
# traj: rows = years, columns = year / pc1 / pc2

from sklearn.decomposition import PCA

TECH_COL = 'technology_meta'
ai_concerns_pca = concerns_df[concerns_df[TECH_COL] == 'AI'].copy()
ai_concerns_pca = ai_concerns_pca.dropna(subset=['year'])
ai_concerns_pca['year'] = ai_concerns_pca['year'].astype(int)

concern_id_to_idx = {cid: i for i, cid in enumerate(concern_ids)}

traj_rows = []
for yr, group in ai_concerns_pca.groupby('year'):
    idxs = [concern_id_to_idx[cid] for cid in group['concern_id'] if cid in concern_id_to_idx]
    if not idxs:
        continue
    avg_emb = concern_embeddings[np.array(idxs)].mean(axis=0)
    traj_rows.append({'year': int(yr), 'embedding': avg_emb})

if traj_rows:
    emb_matrix = np.stack([r['embedding'] for r in traj_rows])
    n_comp = min(2, emb_matrix.shape[0])
    pca = PCA(n_components=n_comp)
    coords = pca.fit_transform(emb_matrix)
    traj = pd.DataFrame({
        'year': [r['year'] for r in traj_rows],
        'pc1': coords[:, 0],
        'pc2': coords[:, 1] if n_comp > 1 else 0.0,
    })
    print(f"traj: {len(traj)} years, explained variance: {pca.explained_variance_ratio_}")
else:
    traj = pd.DataFrame({'year': [], 'pc1': [], 'pc2': []})
    print("Warning: no AI concern embeddings found for trajectory")


This cell generates a visual representation of the evolution of AI-related concerns over time by plotting their displacement in a two-dimensional concern space defined by principal components. By centering the trajectory on the initial year, it allows for an intuitive understanding of how public sentiment has shifted, highlighting trends and patterns in AI concerns across different time periods. This visualization is crucial for identifying temporal dynamics and understanding the societal context of AI discourse, which can inform future research and policy decisions.

In [ ]:
# @title Visualise AI concern trajectory over time

import matplotlib.pyplot as plt
import numpy as np

# Use traj DataFrame from the previous cell
# columns: year, pc1, pc2

traj2 = traj.sort_values("year").reset_index(drop=True)

# Center trajectory on first year
x0, y0 = traj2.loc[0, ["pc1", "pc2"]]
dx = traj2["pc1"] - x0
dy = traj2["pc2"] - y0

years = traj2["year"].to_numpy()

plt.figure(figsize=(6, 6))
sc = plt.scatter(dx, dy, c=years, cmap="viridis", s=80)
plt.plot(dx, dy, alpha=0.6)

plt.axhline(0, color="grey", linewidth=1, alpha=0.5)
plt.axvline(0, color="grey", linewidth=1, alpha=0.5)

plt.xlabel("Displacement in concern space (PC1)")
plt.ylabel("Displacement in concern space (PC2)")
plt.title("AI concern profile over time\n(displacement from initial position)")

cbar = plt.colorbar(sc)
cbar.set_label("Year")

plt.tight_layout()
plt.show()


This cell analyzes the temporal dynamics of AI-related concerns by calculating the relative salience of various concern clusters over time and identifying trends in their prominence. By normalizing the data and applying linear regression to determine the slopes of these trends, it highlights which concerns are gaining or losing attention in public discourse, providing insights into shifting societal priorities regarding AI. This analysis is crucial for understanding how public sentiment evolves and can inform future research and policy-making in the field of AI.

In [ ]:
# @title Compute AI concern year × cluster matrix (ai_year)
# ai_year: rows = years, columns = concern cluster IDs,
# values = FRACTION OF DOCUMENTS in that year mentioning that cluster.
#
# Methodology: CIP-0009 Approach B (document-level binary weighting).
# Each source document is counted at most once per cluster per year,
# regardless of how many phrases it contributes.  The denominator is the
# total number of distinct AI-dialogue documents published that year.
# This removes volume bias (more dialogues in later years) and length bias
# (longer documents containing more phrases).

TECH_COL = 'technology_meta'

ai_year = du.temporal_cluster_frequency(
    phrases_df=concerns_df,
    chunks_df=chunks_df,
    kind='concern',
    tech_filter='AI',
    tech_col=TECH_COL,
)

all_concern_clusters = sorted(concerns_df['cluster_id'].dropna().unique().astype(int).tolist())
ai_year = ai_year.reindex(columns=all_concern_clusters, fill_value=0.0)
years = sorted(ai_year.index.tolist())

print(f"ai_year: {ai_year.shape}  (years × concern clusters for AI, fraction of docs)")
print(ai_year.round(3))


This cell examines the shifts in public discourse regarding AI-related concerns over time by calculating the relative salience of various concern clusters. By normalizing the data and applying linear regression to identify trends, it highlights which concerns are gaining or losing attention, providing valuable insights into the evolving landscape of public sentiment towards AI. Understanding these dynamics is crucial for researchers and policymakers to address emerging issues and align responses with public priorities.

In [ ]:
# @title Analyse AI concern salience trajectories
# Shows which concern clusters rise or fall most within AI discourse over time.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year and years already exist from previous cells
# ai_year: years × clusters (raw salience)

# Normalize within each year so values sum to 1 (relative attention)
ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

# Compute linear trend (slope) for each cluster
# Using simple OLS slope over time
t = np.arange(len(ai_rel))  # time index (no assumption of equal year gaps)
slopes = {}
for c in ai_rel.columns:
    y = ai_rel[c].to_numpy()
    if np.all(y == 0):
        slopes[c] = np.nan
    else:
        # slope of y ~ t
        slopes[c] = np.polyfit(t, y, 1)[0]

trend = pd.Series(slopes).dropna().sort_values(ascending=False)

# Select top rising and declining clusters
N = 8
top_up = trend.head(N)
top_down = trend.tail(N)

# Plot trajectories
plt.figure(figsize=(11, 5))

# Rising
plt.subplot(1, 2, 1)
for c in top_up.index:
    plt.plot(ai_rel.index, ai_rel[c], marker="o", label=c)
plt.title("AI concerns increasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

# Declining
plt.subplot(1, 2, 2)
for c in top_down.index:
    plt.plot(ai_rel.index, ai_rel[c], marker="o", label=c)
plt.title("AI concerns decreasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

plt.tight_layout()
plt.show()

display(
    pd.DataFrame({
        "slope": trend
    }).assign(direction=lambda d: np.where(d["slope"] > 0, "rising", "declining"))
)


This cell generates a heatmap visualizing the relative salience of various AI concern clusters over time, normalized by year to reflect their prominence within public discourse. By highlighting the top concerns and ordering them by their peak years, the analysis provides insights into how public attention to specific AI issues has evolved, which is crucial for understanding temporal dynamics in societal attitudes towards AI technologies. This visualization aids researchers in identifying trends and shifts in public sentiment, informing future dialogue and policy considerations.

In [ ]:
# @title Visualise AI concern salience over time
# Rows = concern clusters
# Columns = years
# Values = relative salience within AI discourse (row-normalised per year)
#
# ai_year holds document-weighted fractions (CIP-0009 Approach B):
# each cell = fraction of AI documents in that year mentioning that cluster.
# We then re-normalise within each year to get the RELATIVE share of attention.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year: years × clusters (doc-fraction per year, from temporal_cluster_frequency)
# years is a sorted array of years

# 1) Normalise within each year (relative attention)
ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

# 2) Select clusters to display
# Option A: top N by overall AI salience
N = 20
top_clusters = ai_rel.sum(axis=0).sort_values(ascending=False).head(N).index

heat = ai_rel[top_clusters]

# 3) Order clusters by when they peak (helps visual interpretation)
peak_year_idx = heat.values.argmax(axis=0)
order = np.argsort(peak_year_idx)
heat = heat.iloc[:, order]

# Transpose so clusters are rows
heat_T = heat.T

# 4) Plot heat map
plt.figure(figsize=(12, 7))
im = plt.imshow(
    heat_T,
    aspect="auto",
    cmap="viridis"
)

plt.colorbar(im, label="Share of AI attention (within year)")

plt.yticks(
    ticks=np.arange(len(heat_T.index)),
    labels=heat_T.index
)
plt.xticks(
    ticks=np.arange(len(heat_T.columns)),
    labels=heat_T.columns,
    rotation=45
)

plt.xlabel("Year")
plt.ylabel("Concern cluster")
plt.title("AI public concerns over time\n(relative salience within AI discourse)")

plt.tight_layout()
plt.show()


This cell imports essential functions for analyzing the temporal dynamics of AI-related discussions, specifically focusing on metrics like normalized entropy, the Herfindahl-Hirschman Index (HHI), and top-k share. These metrics are crucial for quantifying the diversity and concentration of concerns and benefits expressed in public dialogues over time, enabling a nuanced understanding of how societal perceptions of AI evolve.

In [ ]:
# normalized_entropy, hhi, topk_share, parse_year — imported from dialogue_utils (see Cell 5)
pass

This cell visualizes the trajectory of perceived benefits associated with AI over time by plotting the displacements in a two-dimensional benefit space, derived from principal component analysis. By centering the trajectory on the initial year, the analysis highlights how public sentiment regarding AI benefits has evolved, providing insights into temporal trends that can inform future discussions and policy-making in AI development. This visualization is crucial for understanding the dynamics of public dialogue and the shifting perceptions of AI's role in society.

In [ ]:
# @title Compute AI benefit PCA trajectory (benefit_traj)
# benefit_traj: rows = years, columns = year / pc1 / pc2

from sklearn.decomposition import PCA

TECH_COL = 'technology_meta'
ai_benefits_pca = benefits_df[benefits_df[TECH_COL] == 'AI'].copy()
ai_benefits_pca = ai_benefits_pca.dropna(subset=['year'])
ai_benefits_pca['year'] = ai_benefits_pca['year'].astype(int)

benefit_id_to_idx = {bid: i for i, bid in enumerate(benefit_ids)}

benefit_traj_rows = []
for yr, group in ai_benefits_pca.groupby('year'):
    idxs = [benefit_id_to_idx[bid] for bid in group['benefit_id'] if bid in benefit_id_to_idx]
    if not idxs:
        continue
    avg_emb = benefit_embeddings[np.array(idxs)].mean(axis=0)
    benefit_traj_rows.append({'year': int(yr), 'embedding': avg_emb})

if benefit_traj_rows:
    emb_matrix = np.stack([r['embedding'] for r in benefit_traj_rows])
    n_comp = min(2, emb_matrix.shape[0])
    pca = PCA(n_components=n_comp)
    coords = pca.fit_transform(emb_matrix)
    benefit_traj = pd.DataFrame({
        'year': [r['year'] for r in benefit_traj_rows],
        'pc1': coords[:, 0],
        'pc2': coords[:, 1] if n_comp > 1 else 0.0,
    })
    print(f"benefit_traj: {len(benefit_traj)} years, explained variance: {pca.explained_variance_ratio_}")
else:
    benefit_traj = pd.DataFrame({'year': [], 'pc1': [], 'pc2': []})
    print("Warning: no AI benefit embeddings found for trajectory")


This cell generates a visual representation of the evolution of perceived benefits associated with AI across different years, using a two-dimensional displacement plot based on principal component analysis (PCA). By centering the trajectory on the initial year, it allows for a clear comparison of how public sentiment regarding AI benefits has shifted over time, which is crucial for understanding the temporal dynamics of public dialogue and informing future discourse on AI policy and development.

In [ ]:
# @title Visualise AI benefit trajectory over time

import matplotlib.pyplot as plt
import numpy as np

benefit_traj2 = benefit_traj.sort_values("year").reset_index(drop=True)

# Center trajectory on first year
x0, y0 = benefit_traj2.loc[0, ["pc1", "pc2"]]
dx = benefit_traj2["pc1"] - x0
dy = benefit_traj2["pc2"] - y0

benefit_years_arr = benefit_traj2["year"].to_numpy()

plt.figure(figsize=(6, 6))
sc = plt.scatter(dx, dy, c=benefit_years_arr, cmap="viridis", s=80)
plt.plot(dx, dy, alpha=0.6)

plt.axhline(0, color="grey", linewidth=1, alpha=0.5)
plt.axvline(0, color="grey", linewidth=1, alpha=0.5)

plt.xlabel("Displacement in benefit space (PC1)")
plt.ylabel("Displacement in benefit space (PC2)")
plt.title("AI benefit profile over time\n(displacement from initial position)")

cbar = plt.colorbar(sc)
cbar.set_label("Year")

plt.tight_layout()
plt.show()


This cell analyzes the relative salience of AI benefits over time by calculating the linear trends of various benefit categories across public dialogues. By identifying the top rising and declining clusters, it visualizes how public attention to specific AI benefits has shifted, which is crucial for understanding the evolving perceptions of AI's role in society and informing future discourse and policy decisions. The insights gained from these trajectories can help researchers and practitioners anticipate public sentiment and adjust their communication strategies accordingly.

In [ ]:
# @title Compute AI benefit year × cluster matrix (benefit_ai_year)
# benefit_ai_year: rows = years, columns = benefit cluster IDs,
# values = FRACTION OF DOCUMENTS in that year mentioning that cluster.
#
# Same document-level binary weighting as ai_year above (CIP-0009 Approach B).

TECH_COL = 'technology_meta'

benefit_ai_year = du.temporal_cluster_frequency(
    phrases_df=benefits_df,
    chunks_df=chunks_df,
    kind='benefit',
    tech_filter='AI',
    tech_col=TECH_COL,
)

all_benefit_clusters = sorted(benefits_df['cluster_id'].dropna().unique().astype(int).tolist())
benefit_ai_year = benefit_ai_year.reindex(columns=all_benefit_clusters, fill_value=0.0)
benefit_years = sorted(benefit_ai_year.index.tolist())

print(f"benefit_ai_year: {benefit_ai_year.shape}  (years × benefit clusters for AI, fraction of docs)")
print(benefit_ai_year.round(3))


This cell computes and visualizes the trends in the salience of various AI benefits over time, allowing researchers to identify which benefits have gained or lost attention in public dialogues. By calculating the linear slopes of these trends, it highlights the most significant rising and declining themes, which is crucial for understanding shifts in public perception and discourse surrounding AI, ultimately informing future research and policy discussions.

In [ ]:
# @title Analyse AI benefit salience trajectories

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

benefit_ai_rel = benefit_ai_year.div(benefit_ai_year.sum(axis=1), axis=0).fillna(0)

# Compute linear trend (slope) for each cluster
# Note: this uses a year-index, not the actual years, so non-uniform gaps
# are treated as uniform. See methodology limitations.
t = np.arange(len(benefit_ai_rel))
slopes = {}
for c in benefit_ai_rel.columns:
    y = benefit_ai_rel[c].to_numpy()
    if np.all(y == 0):
        slopes[c] = np.nan
    else:
        slopes[c] = np.polyfit(t, y, 1)[0]

trend = pd.Series(slopes).dropna().sort_values(ascending=False)

# Select top rising and declining clusters
N = 8
top_up = trend.head(N)
top_down = trend.tail(N)

# Plot trajectories
plt.figure(figsize=(11, 5))

# Rising
plt.subplot(1, 2, 1)
for c in top_up.index:
    plt.plot(benefit_ai_rel.index, benefit_ai_rel[c], marker="o", label=c)
plt.title("AI benefits increasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

# Declining
plt.subplot(1, 2, 2)
for c in top_down.index:
    plt.plot(benefit_ai_rel.index, benefit_ai_rel[c], marker="o", label=c)
plt.title("AI benefits decreasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

plt.tight_layout()
plt.show()

display(
    pd.DataFrame({
        "slope": trend
    }).assign(direction=lambda d: np.where(d["slope"] > 0, "rising", "declining"))
)


This cell visualizes the relative salience of various AI benefit clusters over time, providing insights into how public discourse around these benefits has evolved. By normalizing the data within each year and focusing on the top clusters, the analysis highlights shifts in public attention, which is crucial for understanding the temporal dynamics of societal perceptions regarding AI and its advantages. This visualization aids researchers in identifying trends and peak periods of interest, informing both academic inquiry and policy discussions related to AI development and communication strategies.

In [ ]:
# @title Visualise AI benefit salience over time
# Rows = benefit clusters
# Columns = years
# Values = relative salience within AI discourse (row-normalised per year)
#
# benefit_ai_year holds document-weighted fractions (CIP-0009 Approach B).
# Re-normalised within each year to get the relative share of attention.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1) Normalise within each year (relative attention)
benefit_ai_rel = benefit_ai_year.div(benefit_ai_year.sum(axis=1), axis=0).fillna(0)

# 2) Select top N clusters by overall AI salience
N = 20
top_clusters = benefit_ai_rel.sum(axis=0).sort_values(ascending=False).head(N).index

heat = benefit_ai_rel[top_clusters]

# 3) Order clusters by when they peak
peak_year_idx = heat.values.argmax(axis=0)
order = np.argsort(peak_year_idx)
heat = heat.iloc[:, order]

heat_T = heat.T

plt.figure(figsize=(12, 7))
im = plt.imshow(
    heat_T,
    aspect="auto",
    cmap="viridis"
)

plt.colorbar(im, label="Share of AI attention (within year)")

plt.yticks(
    ticks=np.arange(len(heat_T.index)),
    labels=heat_T.index
)
plt.xticks(
    ticks=np.arange(len(heat_T.columns)),
    labels=heat_T.columns,
    rotation=45
)

plt.xlabel("Year")
plt.ylabel("Benefit cluster")
plt.title("AI public benefits over time\n(relative salience within AI discourse)")

plt.tight_layout()
plt.show()


This cell generates a heatmap that visualizes the distribution of AI-related concerns across four distinct time periods, highlighting how the prominence of specific concerns has evolved over time. By organizing the data based on when each concern peaked, the analysis reveals patterns of stability and change in public discourse surrounding AI, which is crucial for understanding societal attitudes and the temporal dynamics of AI-related discussions in public dialogues.

In [ ]:
# @title (paper) Figure 4: AI concern profile across four time windows
# Heatmap of cluster shares within each time window for the union of clusters
# that appeared in the top 10 of any window. Rows are ordered by which
# window each cluster reaches its peak in, so the diagonal pattern reads
# as the "content shifts within stable spread" finding.

import json as _json
import numpy as _np
import matplotlib.pyplot as _plt
import matplotlib.colors as _mcolors

PAPER_ASSETS = OUTPUT_FOLDER / "paper_assets"
PAPER_ASSETS.mkdir(exist_ok=True)

# Load traceability and the existing top-clusters-per-window file
_trace = pd.read_csv(OUTPUT_FOLDER / "traceability_paragraphs.csv")
_top_window = pd.read_csv(OUTPUT_FOLDER / "ai_concern_window_top_clusters.csv")
_summary = pd.read_csv(OUTPUT_FOLDER / "cluster_summary.csv")

from pub_dialogue.address import _parse_listcol, assign_window as _assign_window

_ai = _trace[_trace["technology_meta"] == "AI"].copy()
_ai["cluster_ids_parsed"] = _ai["cluster_ids"].apply(_parse_listcol)
_ai["window"] = _ai["year_meta"].apply(_assign_window)

_exploded = _ai[["window", "cluster_ids_parsed"]].explode("cluster_ids_parsed").dropna()
_exploded["cluster_id"] = pd.to_numeric(_exploded["cluster_ids_parsed"], errors="coerce")
_exploded = _exploded.dropna(subset=["cluster_id"])
_exploded["cluster_id"] = _exploded["cluster_id"].astype(int)

_counts = _exploded.groupby(["window", "cluster_id"]).size().reset_index(name="n")
_window_totals = _exploded.groupby("window").size().reset_index(name="window_total")
_counts = _counts.merge(_window_totals, on="window")
_counts["share_within_window"] = _counts["n"] / _counts["window_total"]

_union = _top_window["cluster_id"].unique()
_hm = _counts[_counts["cluster_id"].isin(_union)].copy()

_window_order = ["2004-2017", "2018-2020", "2021-2023", "2024-2025"]
_pivot = _hm.pivot(index="cluster_id", columns="window", values="share_within_window").fillna(0)
_pivot = _pivot[_window_order]

_labels_map = dict(zip(_summary["cluster_id"], _summary["label"]))
_pivot["label"] = _pivot.index.map(_labels_map)

def _peak_window(row):
    shares = row[_window_order]
    return _window_order.index(shares.idxmax())

_pivot["peak_window"] = _pivot.apply(_peak_window, axis=1)
_pivot["peak_share"] = _pivot[_window_order].max(axis=1)
_pivot = _pivot.sort_values(["peak_window", "peak_share"], ascending=[True, False])

# Save the underlying data
_pivot[_window_order + ["label"]].to_csv(PAPER_ASSETS / "figure4_data.csv")

# Render
_data = _pivot[_window_order].values * 100
_labels = _pivot["label"].tolist()
_n = len(_labels)

fig, ax = _plt.subplots(figsize=(8.5, max(7, _n * 0.35)))
_vmax = _np.percentile(_data[_data > 0], 95)
_im = ax.imshow(_data, aspect="auto", cmap=_plt.get_cmap("Blues"),
                norm=_mcolors.Normalize(vmin=0, vmax=_vmax))

ax.set_xticks(range(len(_window_order)))
ax.set_xticklabels(_window_order, fontsize=10)
ax.set_yticks(range(_n))
ax.set_yticklabels(_labels, fontsize=9)
ax.tick_params(axis="x", top=True, bottom=False, labeltop=True, labelbottom=False)

for i in range(_n):
    for j in range(len(_window_order)):
        v = _data[i, j]
        if v == 0:
            ax.text(j, i, "·", ha="center", va="center", fontsize=8.5, color="#bbbbbb")
        else:
            color = "white" if (v / _vmax) > 0.55 else "#222222"
            ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8.5, color=color)
        if v > _vmax:
            ax.text(j + 0.35, i - 0.35, "▲", ha="center", va="center", fontsize=7, color="white")

for i in range(_n + 1):
    ax.axhline(i - 0.5, color="white", linewidth=0.6)
for j in range(len(_window_order) + 1):
    ax.axvline(j - 0.5, color="white", linewidth=0.6)

cbar = _plt.colorbar(_im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Share of AI concern phrases in window (%)", fontsize=9)
cbar.ax.tick_params(labelsize=8)

ax.set_title(f"AI concern profile across four time windows\n"
             f"(union of top-10 clusters from each window; n={_n} clusters)",
             fontsize=11, pad=12)

fig.text(0.02, 0.01,
         "Rows ordered by the time window in which each cluster reaches its peak share. "
         "Cells show share of AI concern phrases within window. · = no phrases. "
         f"▲ = above colour-scale cap of {_vmax:.1f}%.",
         fontsize=7.5, color="#555555", ha="left")

_plt.tight_layout(rect=[0, 0.04, 1, 1])
_plt.savefig(PAPER_ASSETS / "figure4_ai_concerns_across_windows.png",
             dpi=200, bbox_inches="tight", facecolor="white")
_plt.show()
print(f"Saved: {PAPER_ASSETS / 'figure4_ai_concerns_across_windows.png'}")